In [ ]:
import os
from skimage.metrics import structural_similarity
from skimage.metrics import mean_squared_error
import math
import numpy as np
import sirf.Gadgetron as pMR
import matplotlib.pyplot as plt


data_path = '/home/jovyan/devel/SIRF-Contribs/src/notebooks/'
os.chdir(data_path)
import utils
from utils import plot_rpe_3d, plot_rpe_3d_simple

# Can use either of these two as ground truth
ground_truth_with_TV = np.load(os.path.join(data_path, 'ground_truth_with_TV.npy'))
ground_truth_no_TV = np.load(os.path.join(data_path, 'ground_truth_no_TV.npy'))

Nms = 30


def rmse(input_im, ground_truth, border=20):
    if np.shape(input_im)!=np.shape(ground_truth):
        raise ValueError('imgs have diff shape')
    mse = mean_squared_error(np.abs(input_im[border:-border,border:-border,border:-border]), \
        np.abs(ground_truth[border:-border,border:-border,border:-border]))
    return math.sqrt(mse)

def nrmse(input_im, ground_truth, border=20):
    if np.shape(input_im)!=np.shape(ground_truth):
        raise ValueError('imgs have diff shape')
    mse = mean_squared_error(np.abs(input_im[border:-border,border:-border,border:-border]), \
        np.abs(ground_truth[border:-border,border:-border,border:-border]))
    return math.sqrt(mse) / np.max(np.abs(ground_truth))

def ssim(input_im, ground_truth, border=20):
    if np.shape(input_im)!=np.shape(ground_truth):
        raise ValueError('imgs have diff shape')
    res_ssim = structural_similarity(np.abs(input_im[border:-border,border:-border,border:-border]), \
        np.abs(ground_truth[border:-border,border:-border,border:-border]))
    return res_ssim


def read_data(data_path, algo, Nms, ground_truth):
    if algo not in ['FISTA', 'PDHG', 'SPDHG']:
        raise ValueError('algo must be one of FISTA, PDHG, SPDHG')
    res = {'rmse': [], 'nrmse': [], 'ssim': []}
    file_pattern = r'{}_TV_{:02d}ms_it_{:03d}.h5'
    itrobj_fname = os.path.join(data_path, 'recons', f'MS_{Nms}',f'{algo}_itrobj.npy')
    itrobj = np.load(itrobj_fname)
    res['iterations'] = itrobj[0].astype(int)
    res['objective'] = itrobj[1]
    itr = itrobj[0][1:].astype(int)

    for it in itr:
        _fname = os.path.join(data_path, 'recons', f'MS_{Nms}', file_pattern.format(algo, Nms, it) )
        res['rmse'].append( rmse( pMR.ImageData(file=_fname).as_array(), ground_truth) )
        res['nrmse'].append( nrmse( pMR.ImageData(file=_fname).as_array(), ground_truth) )
        res['ssim'].append( ssim( pMR.ImageData(file=_fname).as_array(), ground_truth) )

    return res

In [ ]:
res_pdhg = read_data(data_path, 'PDHG', Nms, ground_truth_with_TV)
res_spdhg = read_data(data_path, 'SPDHG', Nms, ground_truth_with_TV)
res_fista = read_data(data_path, 'FISTA', Nms, ground_truth_with_TV)

In [ ]:

res_pdhg = {'rmse': [], 'nrmse': [], 'ssim': []}
itrobj_fname = os.path.join(data_path, 'recons', f'MS_{Nms}','PDHG_itrobj.npy')
if os.path.exists(itrobj_fname):
    itrobj = np.load(itrobj_fname)
    res_pdhg['iterations'] = itrobj[0].astype(int)
    res_pdhg['objective'] = itrobj[1]
    itr = itrobj[0][1:].astype(int)
else:
    itr = (int(el/Nms) for el in range(1, 10+1))




In [ ]:


file_pattern = r'PDHG_TV_{:02d}ms_it_{:03d}.h5'
for it in itr:
    _fname = os.path.join(data_path, 'recons', f'MS_{Nms}', file_pattern.format(Nms, it) )
    res_pdhg['rmse'].append( rmse( pMR.ImageData(file=_fname).as_array(), ground_truth_with_TV) )
    res_pdhg['nrmse'].append( nrmse( pMR.ImageData(file=_fname).as_array(), ground_truth_with_TV) )
    res_pdhg['ssim'].append( ssim( pMR.ImageData(file=_fname).as_array(), ground_truth_with_TV) )


In [ ]:

res_spdhg = {'rmse': [], 'nrmse': [], 'ssim': []}
file_pattern = r'SPDHG_TV_{:02d}ms_it_{:03d}.h5'
itrobj_fname = os.path.join(data_path, 'recons', f'MS_{Nms}','SPDHG_itrobj.npy')
if os.path.exists(itrobj_fname):
    itrobj = np.load(itrobj_fname)
    res_spdhg['iterations'] = itrobj[0].astype(int)
    res_spdhg['objective'] = itrobj[1]
    itr = itrobj[0][1:].astype(int)
else:
    itr = (int(el * Nms) for el in range(1, 10+1))



In [ ]:


for it in itr:
    _fname = os.path.join(data_path, 'recons', f'MS_{Nms}', file_pattern.format(Nms, it) )
    res_spdhg['rmse'].append( rmse( pMR.ImageData(file=_fname).as_array(), ground_truth_with_TV) )
    res_spdhg['nrmse'].append( nrmse( pMR.ImageData(file=_fname).as_array(), ground_truth_with_TV) )
    res_spdhg['ssim'].append( ssim( pMR.ImageData(file=_fname).as_array(), ground_truth_with_TV) )

In [ ]:

res_fista = {'rmse': [], 'nrmse': [], 'ssim': []}
file_pattern = r'FISTA_TV_{:02d}ms_it_{:03d}.h5'
itrobj_fname = os.path.join(data_path, 'recons', f'MS_{Nms}','FISTA_itrobj.npy')
itrobj = np.load(itrobj_fname)
res_fista['iterations'] = itrobj[0].astype(int)
res_fista['objective'] = itrobj[1]
itr = itrobj[0][1:].astype(int)

for it in itr:
    _fname = os.path.join(data_path, 'recons', f'MS_{Nms}', file_pattern.format(Nms, it) )
    res_fista['rmse'].append( rmse( pMR.ImageData(file=_fname).as_array(), ground_truth_with_TV) )
    res_fista['nrmse'].append( nrmse( pMR.ImageData(file=_fname).as_array(), ground_truth_with_TV) )
    res_fista['ssim'].append( ssim( pMR.ImageData(file=_fname).as_array(), ground_truth_with_TV) )
# res_fista['iterations'] = [i for i in range(15)] + [15, 20,30,40,50,60]
# res_fista['objective'] = [1.69328e-02,                         
# 5.86635e-03,                            
# 4.26685e-03,                           
# 3.17182e-03,                             
# 2.29420e-03,                             
# 1.64270e-03,                             
# 1.20042e-03,                             
# 9.22438e-04,                             
# 7.57521e-04,                             
# 6.61743e-04,                             
# 6.03994e-04,                             
# 5.65786e-04,                             
# 5.37944e-04,                             
# 5.16678e-04,                             
# 5.00539e-04,                             
# 4.88657e-04,
# 4.61891e-04,
# 4.48235e-04,                             
# 4.45248e-04,                             
# 4.44530e-04,                             
# 4.44304e-04]

In [ ]:
# epochs = [it * 10 for it in range(1,11)]
plt.plot(res_pdhg['iterations'][1:], res_pdhg['rmse'], label='PDHG' )
plt.plot([el / Nms for el in res_spdhg['iterations'][1:]], res_spdhg['rmse'], label='SPDHG' )
plt.plot(res_fista['iterations'][1:], res_fista['rmse'], label='FISTA' )
# plt.plot(epochs_spdhg, res_spdhg_all['rmse'], label='SPDHG' )
plt.legend()
plt.title('RMSE')
plt.xlabel('epoch')
plt.show()

In [ ]:
plt.plot(res_pdhg['iterations'][1:], res_pdhg['nrmse'], label='PDHG' )
plt.plot([el / Nms for el in res_spdhg['iterations'][1:]], res_spdhg['nrmse'], label='SPDHG' )
plt.plot(res_fista['iterations'][1:], res_fista['nrmse'], label='FISTA' )
plt.legend()
plt.title('NRMSE')
plt.xlabel('epoch')
plt.show()

In [ ]:
plt.plot(res_pdhg['iterations'][1:], res_pdhg['ssim'], label='PDHG' )
plt.plot([el / Nms for el in res_spdhg['iterations'][1:]], res_spdhg['ssim'], label='SPDHG' )
plt.plot(res_fista['iterations'][1:], res_fista['ssim'], label='FISTA' )
plt.legend()
plt.title('SSIM')
plt.xlabel('epoch')
plt.show()

In [ ]:

offset = 1
scale = 'linear'
plt.plot(res_pdhg['iterations'][offset:], res_pdhg['objective'][offset:], label='PDHG' )
plt.plot([el / Nms for el in res_spdhg['iterations'][offset:]], res_spdhg['objective'][offset:], label='SPDHG' )
plt.plot(res_fista['iterations'][offset:], res_fista['objective'][offset:], label='FISTA' )
plt.xscale(scale)
plt.title(f'objective, {Nms} MS')
plt.xlabel('epoch')
plt.legend()
plt.show()



In [ ]:
num_epochs = 20
_fname = os.path.join(data_path, 'recons', f'MS_{Nms}','SPDHG_TV_{}ms_it_{:03d}.h5'.format(Nms, Nms * num_epochs) )
im = pMR.ImageData(_fname)
plot_rpe_3d([im.as_array(), im.as_array() - ground_truth_with_TV],
            [59, 64] * 2,
            [f'SPDHG, {Nms}ms, {num_epochs} epochs', 'difference with GT'],
            [3*10**(-5)]*2,
            f'SPDHG, {Nms}ms.png')

In [ ]:
_fname = os.path.join(data_path, 'recons', f'MS_{Nms}','PDHG_TV_{}ms_it_{:03d}.h5'.format(Nms, num_epochs) )
im = pMR.ImageData(_fname)
plot_rpe_3d([im.as_array(), im.as_array() -ground_truth_with_TV],
            [59, 64]*2,
            [f'PDHG, {Nms}ms, {num_epochs} epochs', 'difference with GT'],
            [3*10**(-5)]*2,
            f'PDHG, {Nms}ms.png')

In [ ]:
_fname = os.path.join(data_path, 'recons', f'MS_{Nms}','FISTA_TV_{}ms_it_{:03d}.h5'.format(Nms, num_epochs) )
im = pMR.ImageData(_fname)
plot_rpe_3d([im.as_array(), im.as_array() - ground_truth_with_TV],
            [59, 64] * 2,
            [f'FISTA, {Nms}ms, {num_epochs} epochs', 'difference with GT'],
            [3*10**(-5)]*2,
            f'FISTA, {Nms}ms.png')

In [ ]:
plot_rpe_3d([ground_truth_with_TV],
            [59, 64],
            [f'Ground Truth: TV-PDHG, no motion'],
            [3*10**(-5)],
            f'Ground-Truth PDHG, {Nms}ms.png')

In [ ]:
# p1 = self.f(self.operator.direct(self.x)) + self.g(self.x)
import dask
from dask import delayed
import tqdm 
from tqdm import tqdm

def evaluate_objective(algo):
    p1 = 0.
    
    procs = []
    for i,op in enumerate(algo.operator.operators):
        procs.append(delayed(algo.f[i](op.direct)(x)))

    res = dask.compute(*procs)

    for i in range(len(res)):
        p1 += res[i]
    p1 += algo.g(x)

    d1 = - f.convex_conjugate(algo.y_old)
    tmp = algo.operator.adjoint(algo.y_old)
    tmp *= -1
    d1 -= algo.g.convex_conjugate(tmp)

    # self.loss.append([p1, d1, p1-d1])
    return [p1, d1, p1-d1]

In [ ]:
import dask